# 投资组合回测框架使用示例

本 notebook 演示如何使用可扩展的投资组合回测框架

In [2]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np

from portfolio_backtest import (
    BacktestEngine,
    RiskParityStrategy,
    MeanVarianceStrategy
)
from portfolio_backtest.visualization import BacktestVisualizer
from portfolio_backtest.utils import load_price_data, align_columns

import matplotlib.pyplot as plt

from WindPy import w
w.start()


Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2024 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


.ErrorCode=0
.Data=[OK!]

## 1. 加载数据

In [3]:
INDUSTRY_FUND_POOL = [
    "512880.SH",
    "512800.SH",
    "512070.SH",
    "159995.SZ",
    "159819.SZ",
    "515880.SH",
    "159852.SZ",
    "512010.SH",
    "512170.SH",
    "159992.SZ",
    "515170.SH",
    "512690.SH",
    "512400.SH",
    "515220.SH",
    "159870.SZ",
    "512200.SH",
]

INDUSTRY_FUND_POOL_STR = ','.join(INDUSTRY_FUND_POOL)

start_date = "2024-06-01"
end_date = "2025-12-31"


In [4]:
data = w.wsd(INDUSTRY_FUND_POOL_STR, "close", start_date, end_date, "PriceAdj=B")
price_df = pd.DataFrame(data.Data, index=data.Codes, columns=data.Times).T
price_df.index = pd.to_datetime(price_df.index)
price_df

,512880.SH,512800.SH,512070.SH,159995.SZ,159819.SZ,515880.SH,159852.SZ,512010.SH,512170.SH,159992.SZ,515170.SH,512690.SH,512400.SH,515220.SH,159870.SZ,512200.SH
2024-06-03,0.828,1.239000,1.736274,0.841,0.724,1.133,0.589,1.366748,1.001270,0.658,0.610,1.789538,1.044000,2.853201,0.598,0.478000
2024-06-04,0.834,1.247000,1.739242,0.842,0.727,1.140,0.590,1.394083,1.019985,0.673,0.618,1.810874,1.056000,2.807740,0.605,0.487000
2024-06-05,0.831,1.232000,1.736274,0.846,0.724,1.124,0.592,1.394083,1.016866,0.671,0.612,1.789538,1.032000,2.777433,0.602,0.474000
2024-06-06,0.820,1.231000,1.718466,0.843,0.720,1.134,0.578,1.378463,1.001270,0.662,0.607,1.770869,1.042000,2.809905,0.600,0.466000
2024-06-07,0.815,1.240000,1.709562,0.842,0.710,1.106,0.574,1.374558,0.998151,0.658,0.600,1.749534,1.044000,2.833718,0.596,0.477000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-25,1.221,1.637079,2.786943,1.743,1.527,3.140,0.848,1.515138,1.088608,0.856,0.563,1.477502,1.886771,2.422222,0.803,0.534785
2025-12-26,1.229,1.631082,2.798815,1.733,1.523,3.104,0.851,1.511233,1.082370,0.854,0.558,1.466835,1.961343,2.429236,0.820,0.537290
2025-12-29,1.222,1.647073,2.775071,1.735,1.525,3.113,0.856,1.495613,1.076131,0.842,0.554,1.448166,1.918439,2.438588,0.810,0.535143
2025-12-30,1.216,1.647073,2.745391,1.756,1.550,3.115,0.857,1.487803,1.066774,0.837,0.550,1.442832,1.950106,2.419884,0.826,0.529415


In [5]:
# # 加载价格数据
# price_df = pd.read_excel('./market_close.xlsx')
# price_df.columns = price_df.iloc[2]
# price_df = price_df.iloc[4:]
# price_df['日期'] = pd.to_datetime(price_df['日期'])
# price_df = price_df.set_index('日期')
# print(f"数据形状: {price_df.shape}")
# print(f"日期范围: {price_df.index.min()} 到 {price_df.index.max()}")
# print(f"\n资产列表:")
# for col in price_df.columns:
#     print(f"  - {col}")

# price_df.head()

## 1.5. 参数设置

In [6]:
# # 4个大类，大类之间的风险预算分别为：
# # 股票类50%，债券类30%，商品类10%，转债类10%；
# # 在股票类中，A股占60%，美股占40%；
# # 在债券类中，银行二级资本债券占50%，新综合财富(1年以下)占30%，新综合财富(1-3年)占20%；
# # 在商品类中，黄金占50%，原油占50%；
# # 转债类占10%，不区分大类。
# my_risk_budget = {

#     # 共0.5的风险预算分配给股票类，股票类内部按照60%和40%分配
#     '上证指数': 0.15,    # 0.5*0.6*0.5
#     '创业板指': 0.15,    # 0.5*0.6*0.5
#     '纳斯达克指数': 0.1,    # 0.5*0.4*0.5
#     '道琼斯工业平均': 0.1,   # 0.5*0.4*0.5

#     # 转债类占0.1的风险预算，不区分大类
#     '中证转债': 0.1,

#     # 共0.3的风险预算分配给债券类，债券类内部按照50%、30%和20%分配
#     '中债-商业银行二级资本债券财富(总值)指数': 0.15, # 0.3*0.5
#     '中债-新综合财富(1年以下)指数': 0.09,            # 0.3*0.3
#     '中债-新综合财富(1-3年)指数': 0.06,              # 0.3*0.2

#     # 共0.1的风险预算分配给商品类，商品类内部按照50%和50%分配
#     'SGE黄金9999': 0.05,    # 0.1*0.5
#     'ICE布油': 0.05
# }

# if not np.isclose(sum(my_risk_budget.values()), 1.0):
#     raise ValueError("风险预算的总和必须为1.0")

# # 对应起来的资产列表
# my_risk_budget = align_columns(my_risk_budget, price_df.columns)

## 2. 风险平价策略回测

In [7]:
# 创建风险平价策略 - 测试单种方法
rp_strategy = RiskParityStrategy(
    lookback=120,           # 120日回看窗口
    rebalance_freq='ME',    # ME月末调仓, QE 季末调仓
    method='CDD',         # 使用SLSQP优化方法计算权重,也可以选择 'CDD' 方法
    compare_methods=False,   # 关闭方法对比，使用rp_strategy_compare.print_weights_comparison()可以查看权重对比结果
    # risk_budget=my_risk_budget
)

# 创建回测引擎
engine = BacktestEngine(
    init_cash=1_000_000,
    freq='1D'
)

# 运行回测
rp_result = engine.run(rp_strategy, price_df)

In [8]:
# 查看回测统计
print(rp_result.metrics)

rp_result.stats()

{'total_return': np.float64(0.1647073559729997), 'annualized_return': np.float64(0.15465583553287843), 'annualized_volatility': np.float64(0.15342385951569418), 'sharpe_ratio': np.float64(1.0151073151341437), 'sortino_ratio': np.float64(1.3208604149339138), 'calmar_ratio': np.float64(1.2709356051476337), 'max_drawdown': np.float64(-0.12168660230028994), 'omega_ratio': np.float64(1.2099328167468675), 'best_trade': None, 'worst_trade': None, 'win_rate': None}


Start                                 2024-06-03 00:00:00
End                                   2025-12-31 00:00:00
Period                                  387 days 00:00:00
Start Value                                     1000000.0
End Value                                  1164707.355973
Total Return [%]                                16.470736
Benchmark Return [%]                            44.114209
Max Gross Exposure [%]                              100.0
Total Fees Paid                                       0.0
Max Drawdown [%]                                 12.16866
Max Drawdown Duration                   138 days 00:00:00
Total Trades                                         1815
Total Closed Trades                                  1799
Total Open Trades                                      16
Open Trade PnL                               51657.227425
Win Rate [%]                                    58.866037
Best Trade [%]                                  98.450011
Worst Trade [%

In [9]:
rp_result.weights

,512880.SH,512800.SH,512070.SH,159995.SZ,159819.SZ,515880.SH,159852.SZ,512010.SH,512170.SH,159992.SZ,515170.SH,512690.SH,512400.SH,515220.SH,159870.SZ,512200.SH
2024-11-29,0.051219,0.148831,0.051745,0.059068,0.054338,0.054990,0.051416,0.054588,0.051991,0.054274,0.053493,0.049143,0.064874,0.083944,0.060725,0.055361
2024-12-31,0.050659,0.145281,0.050782,0.058482,0.055031,0.055548,0.051968,0.054703,0.052110,0.054863,0.053378,0.048996,0.065339,0.085662,0.061177,0.056021
2025-01-27,0.050050,0.132635,0.049348,0.058800,0.054317,0.053920,0.050639,0.056444,0.053877,0.056932,0.055047,0.050127,0.070726,0.087410,0.063366,0.056362
2025-02-28,0.049472,0.136359,0.048783,0.057755,0.052479,0.052977,0.048953,0.055336,0.052704,0.056126,0.055763,0.050725,0.071817,0.089771,0.064763,0.056216
2025-03-31,0.048564,0.141444,0.047391,0.056230,0.050318,0.051485,0.046618,0.052531,0.050320,0.052676,0.059181,0.053313,0.071757,0.095204,0.064318,0.058650
2025-04-30,0.041198,0.165809,0.044655,0.045706,0.043551,0.047131,0.037407,0.065084,0.051230,0.055542,0.073019,0.063054,0.067409,0.081853,0.064025,0.053326
2025-05-30,0.043541,0.161250,0.046429,0.047060,0.042096,0.043800,0.037108,0.066275,0.050304,0.054361,0.083470,0.073121,0.062380,0.076254,0.060461,0.052090
2025-06-30,0.041602,0.159016,0.044928,0.046123,0.040816,0.042950,0.035476,0.066591,0.050558,0.052895,0.088424,0.077803,0.062483,0.076515,0.061044,0.052775
2025-07-31,0.042671,0.195615,0.045233,0.047431,0.042636,0.045234,0.037988,0.062466,0.047548,0.048789,0.084383,0.075953,0.052540,0.066578,0.055421,0.049514
2025-08-29,0.044230,0.185266,0.047194,0.046401,0.044288,0.043257,0.045354,0.067187,0.052840,0.050801,0.080832,0.071389,0.048438,0.066310,0.054608,0.051602


In [10]:
# 可视化结果
rp_viz = BacktestVisualizer(rp_result)
rp_viz.print_metrics()
rp_viz.plot_summary()


Risk Parity 策略表现
总收益率: 16.47%
年化收益率: 15.47%
年化波动率: 15.34%
夏普比率: 1.015
索提诺比率: 1.321
Calmar比率: 1.271
Omega比率: 1.210
最大回撤: -12.17%



In [11]:
# 权重热力图
rp_viz.plot_weights_heatmap(freq='QE')

In [12]:
# 权重变化分析与调仓点标注
# 分析模型的重要调仓时机
rp_viz.plot_weight_changes_analysis(threshold=0.03)

In [13]:
for item in rp_result.weights.columns:

    rp_viz.plot_assets_and_weights(
        price_df,
        asset_name=item,      # 指定资产名称
        freq='M',               # 按周显示仓位
        weight_alpha=0.3        # 权重柱状图透明度
    )

d:\0信银理财\0512 rp\vectorbt_rp-main\vectorbt_rp-main\portfolio_backtest\visualization\plots.py:419: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



In [14]:
print(rp_result.weights.columns)

for item in rp_result.weights.columns:
    rp_viz.plot_rebalancing_effectiveness(
        price_df,
        asset_name=item,           # 指定资产
    )

Index(['512880.SH', '512800.SH', '512070.SH', '159995.SZ', '159819.SZ',
       '515880.SH', '159852.SZ', '512010.SH', '512170.SH', '159992.SZ',
       '515170.SH', '512690.SH', '512400.SH', '515220.SH', '159870.SZ',
       '512200.SH'],
      dtype='object')


### 如何解读这些图表

**资产走势与权重分析图**：
- **上半部分**：各资产价格走势（归一化到100），帮助理解资产的历史表现
- **下半部分**：对应的权重配置变化，显示模型何时加减仓
- **分析要点**：
  - 当某资产价格下跌时，模型是否增加权重（抄底）或减少权重（止损）
  - 当某资产价格上涨时，模型是否减少权重（获利了结）或增加权重（追涨）
  - 权重变化频率反映策略的调仓灵敏度

**权重变化分析图**：
- 显示所有资产的权重配置时间序列
- 标注重要的调仓点（权重变化超过阈值）
- 统计月度调仓频率，帮助理解策略的活跃度

## 3. 均值方差策略回测

In [15]:
# 创建均值方差策略（最大化夏普比率）
mv_strategy = MeanVarianceStrategy(
    lookback=60,
    rebalance_freq='ME'
)

# 运行回测
mv_result = engine.run(mv_strategy, price_df)

# 可视化
mv_viz = BacktestVisualizer(mv_result)
mv_viz.print_metrics()
mv_viz.plot_summary()


Mean Variance 策略表现
总收益率: 61.97%
年化收益率: 57.59%
年化波动率: 30.47%
夏普比率: 1.645
索提诺比率: 2.560
Calmar比率: 3.331
Omega比率: 1.339
最大回撤: -17.29%



## 4. 策略对比

In [16]:
# 创建多个策略进行对比
strategies = [
    RiskParityStrategy(lookback=60, rebalance_freq='ME',method='CDD'),
    RiskParityStrategy(lookback=120, rebalance_freq='QE',method='CDD'),
    MeanVarianceStrategy(lookback=60, rebalance_freq='ME'),
]

names = ['风险平价(60日/月)', '风险平价(120日/季)', '均值方差(60日/月)']

# 运行所有策略
results = []
for strategy, name in zip(strategies, names):
    result = engine.run(strategy, price_df)
    results.append(result)
    print(f"{name}: 总收益={result.metrics['total_return']*100:.2f}%, 夏普={result.metrics['sharpe_ratio']:.3f}")

风险平价(60日/月): 总收益=44.36%, 夏普=1.628
风险平价(120日/季): 总收益=19.62%, 夏普=1.214
均值方差(60日/月): 总收益=61.97%, 夏普=1.645


In [17]:
# 累计收益对比图
BacktestVisualizer.compare_results(results, names=names)

In [18]:
# 指标对比表
comparison_table = BacktestVisualizer.compare_metrics_table(results, names)
comparison_table

,总收益率 (%),年化收益率 (%),年化波动率 (%),夏普比率,索提诺比率,Calmar比率,最大回撤 (%)
风险平价(60日/月),44.358064,41.376497,22.885371,1.627522,2.540654,2.897603,-14.279558
风险平价(120日/季),19.617748,18.405842,14.842422,1.213668,1.597140,1.599236,-11.509147
均值方差(60日/月),61.974487,57.594148,30.470497,1.644911,2.559814,3.330560,-17.292633


## 5. 创建自定义策略

继承 `BaseStrategy` 即可创建自己的策略

In [19]:
from portfolio_backtest.strategies.base import BaseStrategy

class EqualWeightStrategy(BaseStrategy):
    """等权重策略 - 自定义策略示例"""
    
    def __init__(self, rebalance_freq='ME'):
        super().__init__(name="Equal Weight", rebalance_freq=rebalance_freq)
        self.rebalance_freq = rebalance_freq
    
    def generate_weights(self, price_df, rebalance_mask=None):
        price_df = self.validate_data(price_df)
        
        if rebalance_mask is None:
            rebalance_dates = self.get_rebalance_dates(price_df, self.rebalance_freq)
            rebalance_mask = pd.Series(
                price_df.index.isin(rebalance_dates),
                index=price_df.index
            )
        
        n_assets = price_df.shape[1]
        equal_weight = 1.0 / n_assets
        
        rebalance_dates = price_df.index[rebalance_mask]
        weights_list = [np.full(n_assets, equal_weight) for _ in rebalance_dates]
        
        return pd.DataFrame(
            weights_list,
            index=rebalance_dates,
            columns=price_df.columns
        )

# 使用自定义策略
ew_strategy = EqualWeightStrategy(rebalance_freq='ME')
ew_result = engine.run(ew_strategy, price_df)

ew_viz = BacktestVisualizer(ew_result)
ew_viz.print_metrics()


Equal Weight 策略表现
总收益率: 48.03%
年化收益率: 44.76%
年化波动率: 27.26%
夏普比率: 1.493
索提诺比率: 2.391
Calmar比率: 2.757
Omega比率: 1.281
最大回撤: -16.24%

